Begin by importing necessary packages

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, regexp_replace, explode, array, struct, coalesce, when, trim
from pyspark.sql.types import StructType, IntegerType # Need this for the empty DataFrame above

Set global variables, define filepaths, and create lists of variables

In [3]:
spark = SparkSession.builder \
    .appName("CampusSafetyTrendAnalysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

hdfs_write_root = "hdfs://localhost:9000/data/merged"

# All years present in the dataset (from 2018 to 2023)
ALL_YEARS = ["18", "19", "20", "21", "22", "23"]

# The base institution fields to keep
INSTITUTION_FIELDS = ["UNITID_P", "INSTNM", "OPEID", "BRANCH", "Address", "City", "State", "ZIP", "sector_cd", "Sector_desc", "men_total", "women_total", "Total"]

# Mapping of all possible offense types across all data files
OFFENSE_MAP = {
    "crime": ["MURD", "NEG_M", "RAPE", "FONDL", "INCES", "STATR", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON"],
    "discipline": ["WEAPON", "DRUG", "LIQUOR"],
    "vawa": ["DOMEST", "DATING", "STALK"],
    "hate": ["MURD", "RAPE", "FOND", "INCE", "STAT", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON", "SIM_A", "LAR_T", "INTIM", "VANDAL"],
    # Suffixes for hate crimes are complex, so we'll handle them inside the function
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/03 22:11:47 WARN Utils: Your hostname, lnx1084263govt, resolves to a loopback address: 127.0.1.1; using 10.60.101.143 instead (on interface enp3s0)
25/12/03 22:11:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/03 22:11:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Begin by loading data

In [4]:

try:
    crime_df = spark.read.parquet(f"{hdfs_write_root}/crime.parquet")
    hate_df = spark.read.parquet(f"{hdfs_write_root}/hate.parquet")
    discipline_df = spark.read.parquet(f"{hdfs_write_root}/discipline.parquet")
    vawa_df = spark.read.parquet(f"{hdfs_write_root}/vawa.parquet")

except Exception as e:
    print(f"Error loading parquet: {e}")
    spark.stop()

25/12/03 22:11:50 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/data/merged/crime.parquet.
org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)


Error loading parquet: An error occurred while calling o30.parquet.
: org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
	at ... run in separate thread using org.apache.spark.util.ThreadUtils ... ()
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:814)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:575)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:419)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$

Function to transfrom "wide" format (e.g., MURD18, MURD19, etc.) into a "long" format, where we have a single row per institution, per year, per location, and per offense type.

In [5]:
from pyspark.sql.functions import col, lit, regexp_replace, explode, array, struct, coalesce, when, trim
from pyspark.sql.types import IntegerType, FloatType, StructType

# NOTE: Assume OFFENSE_MAP, ALL_YEARS, and INSTITUTION_FIELDS are defined globally.
# For demonstration, assume they are defined as:
# ALL_YEARS = [f"{i:02d}" for i in range(18, 24)] # ['18', '19', '20', '21', '22', '23']
# INSTITUTION_FIELDS = ["UNITID_P", "INSTNM", "BRANCH", "Address", "City", "State", "ZIP", "sector_cd", "Sector_desc", "men_total", "women_total", "Total", "OPEID"]
# OFFENSE_MAP = {
#     "CRIME": ["MURD", "NEG_M", "RAPE", "FONDL", "INCES", "STATR", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON", "FILTER"],
#     "VAWA": ["DOMEST", "DATING", "STALK", "FILTER"]
# }


def transform_to_long_format(df, data_type, location):
    """
    Transforms a single wide-format DataFrame (e.g., oncampuscrime181920_df)
    into a long-format DataFrame.
    """
    
    offense_codes = OFFENSE_MAP.get(data_type, [])
    
    # 1. Determine all offense columns to pivot
    wide_offense_cols = set()
    kvs = []
    
    # We must determine which years are actually present in the DataFrame's columns
    # instead of relying on a global ALL_YEARS that might contain too many.
    
    # Create a list of all offense-year columns that exist in the DataFrame
    # and build the kvs array simultaneously.
    for year in ALL_YEARS:
        for code in offense_codes:
            col_name = f"{code}{year}"
            if col_name in df.columns:
                wide_offense_cols.add(col_name)
                kvs.append(
                    struct(lit(col_name).alias("Offense_Year"), col(col_name).alias("Count"))
                )

    if not kvs:
        # Proper handling for no columns found
        print(f"Warning: No offense columns found for {data_type}. Returning empty DataFrame.")
        return df.sparkSession.createDataFrame([], StructType([]))
    
    # Identify the base columns (Institution fields)
    base_cols = [c for c in INSTITUTION_FIELDS if c in df.columns]
    
    # 2. Explode to long format
    # a) Select only the base (institution) columns and the exploded key_value struct
    df_long = df.select(
        *base_cols,
        explode(array(*kvs)).alias("key_value")
    ).select(
        *base_cols,
        col("key_value.Offense_Year").alias("Offense_Year"),
        col("key_value.Count").alias("Count")
    ).withColumn("Location", lit(location))

    # 3. Separate Offense Type and Year
    df_final = df_long.withColumn(
        "Year",
        col("Offense_Year").substr(-2, 2)
    ).withColumn(
        "Offense_Code",
        regexp_replace(col("Offense_Year"), r'\\d{2}$', '')
    ).drop("Offense_Year")

    # 4. Final cleanup and conversion
    
    # FIX for Count column: String -> Float -> Integer
    df_final = df_final.withColumn(
        "Count",
        # a) Handle empty strings/nulls by setting to "0.0"
        when(
            (trim(col("Count")).isNull()) | (trim(col("Count")) == lit("")),
            lit("0.0") # Use "0.0" as a string to ensure Float casting works
        ).otherwise(
            coalesce(col("Count"), lit("0.0"))
        ).cast(FloatType()) # b) Casts '2.0' to 2.0 (float) and malformed strings to NULL
    ).withColumn(
        "Count",
        # c) Cast the float column to IntegerType. This safely truncates the .0 suffix.
        col("Count").cast(IntegerType())
    )

    # Convert the two-digit year to a four-digit year (e.g., 23 -> 2023)
    df_final = df_final.withColumn(
        "Year",
        col("Year").cast(IntegerType()) + lit(2000)
    )

    # Final check to set any remaining null Counts (from failed float casts) to 0.
    df_final = df_final.na.fill({"Count": 0}) 

    return df_final

In [6]:
merged_files = [
    ("crime", "ALL_LOCATIONS", f"{hdfs_write_root}/crime.parquet"),
    ("vawa", "ALL_LOCATIONS", f"{hdfs_write_root}/vawa.parquet"),
    ("discipline", "NON_ONCAMPUS", f"{hdfs_write_root}/discipline.parquet"), 
    ("hate", "ALL_LOCATIONS", f"{hdfs_write_root}/hate.parquet"),
]

long_dfs = []
for data_type, location_prefix, path in merged_files:
    print(f"\nProcessing {data_type.upper()} data from {path}...")
    try:
        # Read the merged Parquet file from HDFS
        wide_df = spark.read.parquet(path)
        
        # Transform the wide-format merged DF into the long format
        long_df = transform_to_long_format(wide_df, data_type, location_prefix)
        
        if not long_df.isEmpty():
            long_dfs.append(long_df)
            print(f"Successfully converted {data_type.upper()} to long format.")
            long_dfs[-1].show(5) # https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.show.html
        else:
            print(f"Warning: {data_type.upper()} resulted in an empty long DataFrame.")

    except Exception as e:
        print(f"Error reading or processing {path}: {e}")


Processing CRIME data from hdfs://localhost:9000/data/merged/crime.parquet...
Error reading or processing hdfs://localhost:9000/data/merged/crime.parquet: An error occurred while calling o34.parquet.
: org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
	at ... run in separate thread using org.apache.spark.util.ThreadUtils ... ()
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:814)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:575)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:419)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$Res

25/12/03 22:11:50 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/data/merged/crime.parquet.
org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
25/12/03 22:11:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/data/merged/vawa.parquet.
org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)


Error reading or processing hdfs://localhost:9000/data/merged/vawa.parquet: An error occurred while calling o38.parquet.
: org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
	at ... run in separate thread using org.apache.spark.util.ThreadUtils ... ()
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:814)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:575)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:419)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.sp

25/12/03 22:11:52 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/data/merged/discipline.parquet.
org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
25/12/03 22:11:52 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/data/merged/hate.parquet.
org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)


Error reading or processing hdfs://localhost:9000/data/merged/hate.parquet: An error occurred while calling o46.parquet.
: org.apache.hadoop.ipc.RpcException: RPC response has invalid length of -16777216
	at org.apache.hadoop.ipc.Client$IpcStreams.readResponse(Client.java:1934)
	at org.apache.hadoop.ipc.Client$Connection.receiveRpcResponse(Client.java:1203)
	at org.apache.hadoop.ipc.Client$Connection.run(Client.java:1094)
	at ... run in separate thread using org.apache.spark.util.ThreadUtils ... ()
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:814)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:575)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:419)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.sp

In [7]:
from pyspark.sql import DataFrame
from typing import List

def merge_all_long_format_dfs(df_list: List[DataFrame]) -> DataFrame:
    """
    Merges a list of long-format DataFrames into a single DataFrame using unionByName.
    
    It iteratively unions the DataFrames, starting with the first one.
    
    Args:
        df_list (List[DataFrame]): A list containing the long-format DataFrames 
                                   (e.g., crime_on_campus_long_df, vawa_off_campus_long_df, etc.).
                                   
    Returns:
        DataFrame: A single, merged PySpark DataFrame.
    """
    if not df_list:
        print("Warning: Input list of DataFrames is empty.")
        # Attempt to return an empty DataFrame with a defined schema if possible,
        # but for simplicity and safety, we return None if no schema is available.
        return None 
    
    # Start with the first DataFrame
    merged_df = df_list[0]
    
    # Iterate through the remaining DataFrames and union them by name
    for i, df in enumerate(df_list[1:]):
        # unionByName is crucial to handle potential column order differences, 
        # ensuring columns are matched by name.
        merged_df = merged_df.unionByName(df)
        print(f"Successfully unioned DataFrame {i + 2}.")
        
    return merged_df

# Example Usage (assuming the four long-format DFs exist and have identical schemas):
# all_dfs = [
#     crime_on_campus_long_df, 
#     vawa_on_campus_long_df, 
#     crime_off_campus_long_df, 
#     vawa_off_campus_long_df
# ]
# final_merged_df = merge_all_long_format_dfs(all_dfs)
df = merge_all_long_format_dfs(long_dfs)
df.show()

AttributeError: 'NoneType' object has no attribute 'show'

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql import SparkSession

# Assuming master_df contains the union of all long-format DataFrames (Crime, VAWA, Discipline, Hate).
# If you haven't created master_df yet, you must run that union step first.

ALL_YEARS_4DIGIT = [2018, 2019, 2020, 2021, 2022, 2023]

# Create the final aggregated trend for display
trend_result = df.groupBy("Year").agg( 
    spark_sum("Count").alias("TotalOffenses")
).orderBy("Year")

print("\n--- ✅ Final Trend Validation: Total Offenses by Year ---")

# Display the results
trend_result.show(n=20)

In [ ]:
from pyspark.sql.functions import col

# ------------------------------------------------------------
# 1. Choose a source DataFrame that includes the year '21'
# We use 'oncampuscrime212223_df' which should contain the columns RAPE21, MURD21, etc.
# ------------------------------------------------------------

ALL_YEARS_4DIGIT = [2018, 2019, 2020, 2021, 2022, 2023]

# List of your final merged DataFrames and their names

# --------------------------------

print("--- 🔎 Starting Full Iterative Data & Count Check ---")
print("--- Check focuses on records where Count >= 0 ---")
print("-" * 50)

for df in long_dfs:
    # print(f"\n### Data Check for {df_name} ###")
    
    # Calculate total records for the entire DataFrame once
    total_records = df.count()
    # print(f"Total Records in {df_name} (All Years): {total_records}")

    yearly_counts = []
    
    for year in ALL_YEARS_4DIGIT:
        # 1. Filter the DataFrame for the current year
        data_by_year = df.filter(col("Year") == year)
        
        # 2. Get the count of rows for that year
        year_count = data_by_year.count()
        yearly_counts.append((year, year_count))

        # 3. Check for the specific issue of a truly missing year (Count = 0)
        if year_count == 0:
            print(f"🚨 **WARNING: Year {year} has NO records.**")

    # Display a summary table for clarity
    print("\nYearly Record Count Summary:")
    
    # Use a display-friendly format (or pandas for better table output if available)
    print("+------+-------------------+")
    print("| Year | Total Records     |")
    print("+------+-------------------+")
    for year, count in yearly_counts:
        status = "⚠️ ZERO" if count == 0 else ""
        print(f"| {year} | {count:<17} | {status}")
    print("+------+-------------------+")

Total Crime Count by Year Visualization

In [ ]:
# Group only by Year (Correct aggregation for the single trend line)
trend_result = df.groupBy("Year").agg( 
    spark_sum("Count").alias("TotalOffenses")
).orderBy("Year")

In [ ]:
# Assuming 'trend_result' is the aggregated Spark DataFrame
# WARNING: Only use .toPandas() on SMALL, aggregated DataFrames!
import pandas as pd

if trend_result.count() > 0:
    # This transfers the small data set from the cluster to the local machine memory
    trend_pandas_df = trend_result.toPandas() 
    
    print("\n--- Pandas DataFrame Ready for Graphing ---")
    print(trend_pandas_df.head())
else:
    print("Trend result is empty, cannot plot.")
    # Exit or handle error

In [ ]:
import matplotlib.pyplot as plt

# Check if the Pandas DataFrame exists and is not empty
if 'trend_pandas_df' in locals() and not trend_pandas_df.empty:
    
    plt.figure(figsize=(10, 6)) # Set size for better readability
    
    # Create the line plot
    plt.plot(
        trend_pandas_df["Year"], 
        trend_pandas_df["TotalOffenses"], 
        marker='o', # Add circles to show data points
        linestyle='-', 
        color='blue'
    )
    
    # Add labels and title
    plt.title("Annual Total Offenses", fontsize=16)
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Total Number of Offenses", fontsize=12)
    plt.xticks(trend_pandas_df["Year"]) # Ensure all years are shown on the x-axis
    plt.grid(True, linestyle='--', alpha=0.6) # Add a subtle grid
    
    # Optional: Annotate each data point
    for i, row in trend_pandas_df.iterrows():
        plt.annotate(
            f'{row["TotalOffenses"]:,}', 
            (row["Year"], row["TotalOffenses"]), 
            textcoords="offset points", 
            xytext=(0, 10), 
            ha='center'
        )

    # Display the plot
    plt.tight_layout()
    plt.show()
    print("Graph of Total Offenses by Year has been generated.")

In [ ]:
from pyspark.sql.functions import coalesce, lit, sum as spark_sum, col
from pyspark.sql.types import FloatType, LongType
from functools import reduce 

# NOTE: OFFENSE_CODES is defined above (in user prompt) but must be available here.
OFFENSE_CODES  = ["MURD", "NEG_M", "RAPE", "FONDL", "INCES", "STATR", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON"]

# ============================================================
# 2. TRANSFORM DATA: PIVOT AND SUM OFFENSES BY YEAR (FIXED)
# ============================================================

year_total_exprs = []

for y in ALL_YEARS:
    yearly_crime_cols = [f"{code}{y}" for code in OFFENSE_CODES]

    # FIX: Explicitly cast the column value to FloatType (FloatType is safe for '0.0' strings) 
    # and coalesce with 0.0 before starting the sum operation.
    coalesced_cols = [coalesce(col(col_name).cast(FloatType()), lit(0.0)) 
                      for col_name in yearly_crime_cols 
                      if col_name in crime_df.columns]

    if coalesced_cols:
        # Calculate the sum ACROSS columns (horizontally) using `reduce` with the '+' operator.
        yearly_sum_expr = reduce(lambda a, b: a + b, coalesced_cols)
        
        # Cast the final horizontal sum back to Long/Integer since crime counts are whole numbers
        year_total_exprs.append(
            yearly_sum_expr.cast(LongType()).alias(f"TotalCrime_{y}")
        )

# Select the base columns and the new calculated columns.
crime_trends_by_inst = crime_df.select(*INSTITUTION_FIELDS, *year_total_exprs)

print("Schema of aggregated trends (one row per institution):")
crime_trends_by_inst.printSchema()

# ============================================================
# 3. CALCULATE NATIONAL (or STATE) TRENDS
# ============================================================

# Aggregate by State (Summing the pre-calculated TotalCrime_Y columns vertically)
national_crime_totals = crime_trends_by_inst.groupBy("State").agg(
    *[spark_sum(f"TotalCrime_{y}").alias(f"Total_{y}") for y in ALL_YEARS]
)

print("National/State-level crime totals:")
national_crime_totals.orderBy("State").show()

In [ ]:
from pyspark.sql.functions import sum as spark_sum, col, lit, when
from pyspark.sql import DataFrame
from typing import List
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Re-defining the merge function and 'df' to restore the correct long-format table
def merge_all_long_format_dfs(df_list: List[DataFrame]) -> DataFrame:
    if not df_list:
        return None 
    
    from functools import reduce
    # Use reduce and unionByName to combine all long-format DataFrames
    merged_df = reduce(
        lambda df1, df2: df1.unionByName(df2), 
        df_list
    )
    return merged_df

# Re-merge the DataFrames to ensure 'df' is the long-format master table
# Assuming 'long_dfs' is the list of long-format DataFrames previously created
df = merge_all_long_format_dfs(long_dfs)


# ============================================================
# 1. OFFENSE TYPE TREND ANALYSIS (LINE GRAPH)
# ============================================================

# Using ALL_YEARS from the notebook context: ["18", "19", "20", "21", "22", "23"]
ALL_YEARS = [f"{i:02d}" for i in range(18, 24)]

print("--- Calculating Trend by Offense Type and Year ---")

# 1. Aggregate by Offense Code and Year, excluding generic 'FILTER' codes and zero counts
offense_trend_df = df.filter(
    (col("Offense_Code") != "FILTER") & (col("Count") > 0)
).groupBy("Year", "Offense_Code").agg(
    spark_sum("Count").alias("TotalCount")
)

# 2. Identify the Top 7 most frequent offenses across all years
total_offenses_by_code = offense_trend_df.groupBy("Offense_Code").agg(
    spark_sum("TotalCount").alias("GrandTotal")
).orderBy(col("GrandTotal").desc()).limit(7)

top_n_codes = [row['Offense_Code'] for row in total_offenses_by_code.collect()]

# 3. Filter the trend data to only include the Top 7 offenses
final_trend_df = offense_trend_df.filter(col("Offense_Code").isin(top_n_codes)).orderBy("Year")

# 4. Convert to Pandas for Matplotlib plotting
pdf_offense_trend = final_trend_df.toPandas()

# 5. Create the plot
plt.figure(figsize=(12, 7))

for code in top_n_codes:
    subset = pdf_offense_trend[pdf_offense_trend['Offense_Code'] == code]
    plt.plot(subset['Year'], subset['TotalCount'], marker='o', label=code)

plt.title("Trend of Top 7 Campus Offenses (2018-2023)", fontsize=16)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Total Count of Offenses", fontsize=12)
plt.xticks(pdf_offense_trend['Year'].unique())
plt.legend(title="Offense Code", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("top_offense_trends.png")
print("Trend analysis graph saved to top_offense_trends.png")


# ============================================================
# 2. CHOROPLETH MAP VISUALIZATION
# ============================================================

print("\n--- Aggregating Data for Choropleth Map ---")

# 1. Aggregate the FULL merged DF to create a wide DataFrame: Total Crime per State per Year
# Create the sum expressions dynamically, matching the 4-digit year from the 'Year' column
year_total_exprs = [
    spark_sum(when(col("Year") == (2000 + int(y)), col("Count")).otherwise(0)).alias(f"Total_{y}") 
    for y in ALL_YEARS
]

# Group by State, summing the yearly totals (excluding FILTER codes)
state_crime_totals = df.filter(col("Offense_Code") != "FILTER").groupBy("State").agg(*year_total_exprs)

# 2. Convert Spark DF to Pandas DF for Plotly
pdf = state_crime_totals.toPandas()

# 3. Prepare data for plotting (Unpivoting)
pdf["State"] = pdf["State"].str.upper() # Ensure consistent state codes
year_cols = [f"Total_{y}" for y in ALL_YEARS]

pdf_long = pdf.melt(
    id_vars="State",
    value_vars=year_cols,
    var_name="Year",
    value_name="TotalCrime"
)

# Convert Year suffix (e.g., '18') to 4-digit year string (e.g., '2018')
pdf_long["Year"] = pdf_long["Year"].str.replace("Total_", "").apply(lambda x: f"20{x}")

# 4. Create Choropleth Map (Animated by Year)
fig = px.choropleth(
    pdf_long,
    locations="State",
    locationmode="USA-states", 
    color="TotalCrime",
    scope="usa",
    animation_frame="Year",
    hover_name="State",
    hover_data={"TotalCrime": True, "Year": False},
    color_continuous_scale="Reds",
    labels={"TotalCrime": "Total Offenses (CRIME, VAWA, DISC, HATE)", "Year": "Year"},
    title="Total Campus Offenses by State and Year (2018-2023)"
)

fig.write_json("campus_crime_choropleth.json")
print("Choropleth map JSON saved to campus_crime_choropleth.json")

In [ ]:
import time
from pyspark.sql.functions import sum as spark_sum, col

# --- 1. Define the benchmark function ---
df.show()

def benchmark_aggregation(df, num_partitions):
    """
    Runs a sample aggregation job after repartitioning the DataFrame,
    and measures the total execution time.
    """
    print(f"\n--- Running with {num_partitions} partitions ---")
    
    # Repartition the DataFrame to the desired number of partitions
    # .repartition() is often key to testing the pooling strategy
    repartitioned_df = df.repartition(num_partitions)
    
    # Define a simple aggregation job (replace with your actual target operation)
    # The 'Count' column is used for aggregation here, grouped by 'Year'
    repartitioned_df.show()
    test_job = repartitioned_df.groupBy(col("INSTM")).agg( 
        spark_sum("Count").alias("TotalOffenses")
    )
    
    # Start the timer
    start_time = time.time()
    
    # Trigger the action (e.g., writing the result or collecting data)
    # .collect() forces computation but transfers data to the driver (use with care!)
    # A safer, more production-like action is .count() or writing to HDFS.
    try:
        # Action: Force computation and count rows (fastest way to trigger the DAG)
        result = test_job.count() 
        print(f"Resulting DataFrame has {result} rows.")
        
    except Exception as e:
        print(f"Job failed: {e}")
        result = 0

    # Stop the timer
    end_time = time.time()
    
    # Calculate and return the elapsed time
    elapsed_time = end_time - start_time
    print(f"Time taken: {elapsed_time:.2f} seconds.")
    
    return elapsed_time

# --- 2. Define Pooling Test Cases ---

# Test Case A: Optimal (e.g., 2-3 times the number of cores in your cluster)
# This represents a "not pooled" strategy (many partitions)
large_pool_partitions = 16 

# Test Case B: Pooled Strategy (often a small, fixed number)
# This represents the "pooled" strategy (few partitions)
small_pool_partitions = 5

# --- 3. Execute Benchmarks ---

if 'df' in locals():
    
    # Clean up cache from previous runs to ensure fair comparison
    df.sparkSession.catalog.clearCache()

    # Run the large pool (not pooled) test
    time_large = benchmark_aggregation(df, large_pool_partitions)

    # Clean up cache again
    df.sparkSession.catalog.clearCache()

    # Run the small pool (pooled) test
    time_small = benchmark_aggregation(df, small_pool_partitions)

    # --- 4. Conclusion ---
    print("\n--- Summary ---")
    print(f"Large Pool ({large_pool_partitions} partitions): {time_large:.2f} seconds")
    print(f"Small Pool ({small_pool_partitions} partitions): {time_small:.2f} seconds")

    if time_small < time_large:
        print("\n✅ Conclusion: The smaller pool (16 partitions) was faster. Consider *pooling* your data for this operation.")
    else:
        print("\n✅ Conclusion: The larger pool (100 partitions) was faster. A higher degree of parallelism is likely optimal.")

else:
    print("Error: The DataFrame 'df' is not defined. Please ensure the merge step has run successfully.")

Total crime count over time and crime percent change over time 

In [ ]:
import matplotlib.pyplot as plt

# The DataFrame trend_analysis_df should be defined from the previous PySpark cell.
pdf_trend = trend_analysis_df.orderBy("Year").toPandas()
pdf_trend["Year"] = pdf_trend["Year"].astype(int)

# Create two subplots stacked vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
fig.suptitle("National Campus Crime Trend and Year-over-Year Change", fontsize=14)

# ---- Plot 1: Total crime count by year (Line) ----
ax1.plot(pdf_trend["Year"], pdf_trend["TotalCrimeCount"], marker="o", color='#8B0000', linewidth=2)
ax1.set_ylabel("Total Crime Count")
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# ---- Plot 2: YoY % change (Bar) ----
ax2.bar(pdf_trend["Year"], pdf_trend["YoY_Change_Percent"], color='skyblue')
ax2.axhline(0, linestyle="--", color='gray') # Zero line is essential for YoY clarity
ax2.set_xlabel("Year")
ax2.set_ylabel("YoY Change (%)")
ax2.grid(axis='y', linestyle='--', alpha=0.6)

plt.xticks(pdf_trend["Year"])
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle
plt.show()

In [ ]:
# Stop Spark session
spark.stop()